<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 8 · Performance Measurement, Backtesting, and Pitfalls

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook extends Chapter 8 by:
- computing core performance metrics (annualized return/volatility, Sharpe, drawdown),
- implementing a modular backtest loop, and
- visualizing equity curves, rolling Sharpe, and drawdowns.


### Getting Help While Evaluating Performance
- **Chapter 2** provides statistical background for return aggregation.
- **Chapter 3** explains pandas plotting basics.
- **Appendix B** lists NumPy/pandas snippets for resampling and rolling calculations.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"


## 1. Load Returns and Aggregate Statistics
We reuse the <code>pyaiam_eod.csv</code> dataset to compute daily log returns for the main equity trio.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=["Date"]).set_index("Date").sort_index().ffill()
assets = ["AAPL", "NVDA", "SPY"]
log_rets = np.log(prices[assets] / prices[assets].shift(1)).dropna()
summary = pd.DataFrame(
    {
        "avg_return": log_rets.mean() * 252,
        "vol": log_rets.std() * np.sqrt(252),
    }
)
summary

### 1.1 Portfolio Metrics Helper
A reusable function keeps metric calculations consistent across strategies.

In [ ]:
def performance_stats(returns: pd.Series, risk_free: float = 0.02):
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (ann_ret - risk_free) / ann_vol if ann_vol > 0 else np.nan
    wealth = (1 + returns).cumprod()
    max_dd = (wealth / wealth.cummax() - 1).min()
    return pd.Series(
        {
            "annualized_return": ann_ret,
            "annualized_vol": ann_vol,
            "sharpe": sharpe,
            "max_drawdown": max_dd,
        }
    )

perf_example = performance_stats(log_rets["AAPL"])
perf_example

## 2. Modular Backtest Skeleton
We separate signal generation, portfolio construction, and accounting for clarity.

In [ ]:
def generate_signal(data: pd.DataFrame) -> pd.Series:
    momentum = data.pct_change(21).iloc[:, 0]
    return np.sign(momentum).shift(1).dropna()

def backtest(prices: pd.Series) -> pd.Series:
    sig = generate_signal(prices.to_frame())
    aligned_prices = prices.loc[sig.index]
    rets = aligned_prices.pct_change().dropna()
    strategy_rets = rets * sig.loc[rets.index]
    return strategy_rets

strategy_returns = backtest(prices["AAPL"])
strategy_returns.head()

### 2.1 Visualization: Equity Curve & Rolling Sharpe

We turn the raw return series into a cumulative equity curve and a rolling Sharpe ratio so you can see when the strategy earns and loses money over time.

In [ ]:
def plot_performance(returns: pd.Series, title: str):
    wealth = (1 + returns).cumprod()
    rolling_sharpe = returns.rolling(63).mean() / returns.rolling(63).std()
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    wealth.plot(ax=axes[0])
    axes[0].set_title(f"Equity Curve – {title}")
    axes[0].set_ylabel("Growth of $1")
    rolling_sharpe.plot(ax=axes[1])
    axes[1].set_title("Rolling 3M Sharpe (daily approximation)")
    axes[1].set_ylabel("Sharpe")
    plt.show()

plot_performance(strategy_returns, "Momentum Tilt")

## 3. Drawdown Analysis

Here we derive the drawdown series from the equity curve and visualize peak‑to‑trough losses, which is often more intuitive for stakeholders than volatility alone.

In [ ]:
def drawdown_series(returns: pd.Series) -> pd.Series:
    wealth = (1 + returns).cumprod()
    return wealth / wealth.cummax() - 1

fig, ax = plt.subplots(figsize=(12, 6))
drawdown_series(strategy_returns).plot(ax=ax)
ax.set_title("Drawdown Profile – Momentum Tilt")
ax.set_ylabel("Drawdown")
plt.show()

## 4. Exercises
### Exercise 1 – Benchmark Comparison
Build a function that compares strategy metrics against a buy-and-hold benchmark (SPY).
<details><summary>Hint</summary>
Call <code>performance_stats</code> on both series and subtract key metrics.
</details>

### Exercise 2 – Transaction Costs
Modify the backtest to deduct 5 bps every time the signal changes sign.
<details><summary>Hint</summary>
Track <code>sig.diff().ne(0)</code> and subtract cost * indicator.
</details>

### Exercise 3 – Rolling Windows
Re-run the momentum backtest using rolling 6-month windows to mimic walk-forward testing.
<details><summary>Hint</summary>
Slice the price series with <code>rolling_window</code> or manual date loops, collecting stats per window.
</details>


## 5. Takeaways for Chapter 8
- Consistent metric helpers reduce copy/paste errors.
- Modular backtests make it easier to test alternative signals.
- Visual diagnostics (equity curves, rolling Sharpe, drawdowns) complement scalar metrics.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">